# PhenoRewire Tutorial

This notebook walks through a complete PhenoRewire analysis on a synthetic metabolomics dataset with **known ground truth**. By the end you will know:

1. What inputs PhenoRewire expects
2. How to configure and run the pipeline
3. How to read the key outputs
4. How to interpret rewiring scores, sign-switch edges, and triage rankings

**No data upload needed** — the example dataset ships with the repository in `examples/`.

## 0. Setup

Make sure PhenoRewire is installed:

```bash
pip install -e .               # from repo root
pip install phenorewire[viz]   # optional: adds matplotlib
```

We also need scikit-learn for the benchmark metrics below:

```bash
pip install scikit-learn
```

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# Locate repo root — works whether the notebook is opened from fixtures/ or repo root
_here = Path().resolve()
REPO_ROOT = _here
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Repo root:", REPO_ROOT)

---

## 1. Explore the input data

The example dataset has:
- **22 features** (metabolites) — F001–F022
- **20 samples** — 10 reference, 10 case
- Known biological structure:
  - F001–F004: shared between groups (Module A, energy metabolism)
  - F005–F008: reference-specific correlations (Module B)
  - F009–F012: case-specific correlations (Module C)
  - F013–F015: strongly differentially abundant (Module D)
  - F016–F017: **sign-switch** — positive correlation in reference, negative in case
  - F020–F022: pure noise

In [ ]:
EXAMPLE_DIR = REPO_ROOT / "data"

features = pd.read_csv(EXAMPLE_DIR / "feature_table.csv")
metadata = pd.read_csv(EXAMPLE_DIR / "metadata.csv")

print(f"Feature table: {features.shape[0]} features x {features.shape[1]} columns")
print(f"Metadata:      {metadata.shape[0]} samples")
print()
print("Group distribution:")
print(metadata["group_label"].value_counts().to_string())
print()
features.head(5)

In [ ]:
# Intensity columns are named after sample IDs (e.g. "Sample_01")
intensity_cols = [c for c in features.columns if c.startswith("Sample_")]
print(f"Intensity columns detected: {len(intensity_cols)}")
print(intensity_cols[:5], "...")

---

## 2. Configure the pipeline

PhenoRewire is configured with a `LeanConfig` object (or a YAML file for the CLI).

Key parameters to understand:

| Parameter | What it controls |
|---|---|
| `FDR_ALPHA` | Stringency of feature selection (lower = fewer, more confident features) |
| `CORR_ABS_THRESHOLD` | Minimum |r| to retain a network edge |
| `SIGN_SWITCH_MIN_R` | Minimum |r| in both groups before an edge is flagged as sign-switched |
| `TRIAGE_TOPOLOGY_WEIGHT` | How much network topology (hub status, betweenness) counts vs. phenotype association |
| `ADAPTIVE_SELECTION_THRESHOLDS` | **Opt-in only.** Relaxes FDR when too few features pass. Always leave False unless needed. |

In [ ]:
import tempfile, os
from pathlib import Path

from phenorewire.config import LeanConfig

# Write input files to temp paths so paths are absolute
tmp = Path(tempfile.mkdtemp())
feat_path = tmp / "feature_table.csv"
meta_path = tmp / "metadata.csv"
outdir     = tmp / "output"
outdir.mkdir()

features.to_csv(feat_path, index=False)
metadata.to_csv(meta_path, index=False)

config = LeanConfig(
    DATA_MATRIX=str(feat_path),
    METADATA=str(meta_path),
    OUTDIR=str(outdir),
    ANALYSIS_MODE="phenotype",
    GROUP_DEFINITION={
        "Reference": {"group_label": "reference"},
        "Case":      {"group_label": "case"},
    },
    PHENO_GROUP_REF="Reference",
    PHENO_GROUP_CASE="Case",
    FEATURE_ID_COL="feature_id",
    MZ_COL="mz",
    RT_COL="rt",
    NAME_COL="name",
    META_SAMPLE_COL="sample_id",
    INTENSITY_REGEX=r"^Sample_",
    N_PERMUTATIONS_PHENO=200,
    FDR_ALPHA=0.2,
    ADAPTIVE_SELECTION_THRESHOLDS=False,  # leave off unless data is sparse
    MIN_FEATURES_HARD_STOP=3,
    CORR_ABS_THRESHOLD=0.4,
    CORR_FDR_ALPHA=0.2,
    ADAPTIVE_NETWORK_THRESHOLDS=True,
    NETWORK_MIN_EDGES=3,
    NETWORK_MIN_EDGES_HARD_STOP=3,
    NETWORK_MIN_NODES_HARD_STOP=3,
    SIGN_SWITCH_MIN_R=0.4,
    LOUVAIN_STABILITY_CHECK=False,
    TRIAGE_TOPOLOGY_WEIGHT=0.7,
    TRIAGE_SELECTION_WEIGHT=0.3,
    LOG_TRANSFORM=True,
    NORM_METHOD="median",
    RANDOM_SEED=42,
)

print("Config validated successfully.")
print(f"  FDR_ALPHA:              {config.FDR_ALPHA}")
print(f"  CORR_ABS_THRESHOLD:     {config.CORR_ABS_THRESHOLD}")
print(f"  SIGN_SWITCH_MIN_R:      {config.SIGN_SWITCH_MIN_R}")
print(f"  TRIAGE_TOPOLOGY_WEIGHT: {config.TRIAGE_TOPOLOGY_WEIGHT}")
print(f"  RANDOM_SEED:            {config.RANDOM_SEED}")

---

## 3. Run the pipeline

In [ ]:
from phenorewire.run import run

run(config)
print("Pipeline complete.")

---

## 4. Read the outputs

### 4.1 Final report (narrative summary)

In [ ]:
report_path = outdir / "report" / "final_report.md"
if not report_path.exists():
    # Fallback: some runs write the report directly to triage/
    report_path = outdir / "triage" / "final_report.md"

print(report_path.read_text(encoding="utf-8"))

### 4.2 Rewiring summary

The one-line global summary tells you how much of the network changed between groups.

In [ ]:
summary_path = outdir / "triage" / "rewiring_summary.csv"
if not summary_path.exists():
    # Alternative location
    summary_path = next(outdir.rglob("rewiring_summary.csv"), None)

if summary_path and summary_path.exists():
    rw_summary = pd.read_csv(summary_path)
    display(rw_summary)
else:
    print("rewiring_summary.csv not found — check OUTDIR contents.")

**Key columns to read:**
- `rewiring_proportion` — fraction of all network edges that changed state. Values near 0 = stable network; near 1 = wholesale restructuring.
- `sign_switch_proportion` — fraction of *shared* edges where the sign of correlation flipped. High values suggest antagonistic regulation changes.
- `n_sign_switch` — absolute count of sign-switched edges.

### 4.3 Priority rewired nodes

This is the main result: which metabolites show the most rewired connectivity?

In [ ]:
priority_path = outdir / "triage" / "priority_rewired_nodes.csv"
if not priority_path.exists():
    priority_path = next(outdir.rglob("priority_rewired_nodes.csv"), None)

if priority_path and priority_path.exists():
    rewired = pd.read_csv(priority_path)
    print(f"Top rewired nodes ({len(rewired)} total):")
    display(rewired.head(15))
else:
    print("priority_rewired_nodes.csv not found.")

**Column guide:**

| Column | Meaning |
|---|---|
| `rewiring_score` | Total changed edges (reference-only + case-only + sign-switched) |
| `rewiring_fraction` | Fraction of node's connections that changed (0 = conserved, 1 = fully rewired) |
| `ref_only_edges` | Edges present in reference network only |
| `case_only_edges` | Edges present in case network only |
| `sign_switch_edges` | Edges shared but with inverted correlation sign |

Nodes with high `sign_switch_edges` are particularly interesting — they indicate relationships that inverted direction, not just appeared or disappeared.

### 4.4 Ground-truth recovery check

We know which features were injected as rewired in the synthetic data. Let us verify the pipeline recovers them.

In [ ]:
# Known ground truth from generate_example_data.py:
# F005-F012 have group-specific correlations (rewired connectivity)
# F016, F017 are sign-switched
GROUND_TRUTH_REWIRED = {"F005", "F006", "F007", "F008",
                         "F009", "F010", "F011", "F012",
                         "F016", "F017"}
GROUND_TRUTH_SIGN_SWITCH = {"F016", "F017"}

if priority_path and priority_path.exists():
    rewired = pd.read_csv(priority_path)
    top_n = 10
    top_ids = set(rewired.head(top_n)["feature_id"].astype(str).tolist())
    overlap = top_ids & GROUND_TRUTH_REWIRED

    print(f"Top-{top_n} ranked nodes: {sorted(top_ids)}")
    print(f"Ground-truth rewired:    {sorted(GROUND_TRUTH_REWIRED)}")
    print(f"Overlap:                 {sorted(overlap)}")
    print(f"Precision@{top_n}:            {len(overlap)/top_n:.0%}")

    # Sign-switch recovery
    if "sign_switch_edges" in rewired.columns:
        detected_ss = set(
            rewired.loc[rewired["sign_switch_edges"] > 0, "feature_id"]
            .astype(str).tolist()
        )
        ss_overlap = detected_ss & GROUND_TRUTH_SIGN_SWITCH
        print(f"\nSign-switch nodes detected: {sorted(detected_ss)}")
        print(f"Ground-truth sign-switch:   {sorted(GROUND_TRUTH_SIGN_SWITCH)}")
        print(f"Sign-switch recall:         {len(ss_overlap)/len(GROUND_TRUTH_SIGN_SWITCH):.0%}")

### 4.5 AUROC (quantitative recovery metric)

In [ ]:
try:
    from sklearn.metrics import roc_auc_score, average_precision_score
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print("scikit-learn not installed — skip AUROC. Run: pip install scikit-learn")

if HAS_SKLEARN and priority_path and priority_path.exists():
    rewired = pd.read_csv(priority_path)
    all_ids = rewired["feature_id"].astype(str).tolist()
    scores  = rewired["rewiring_score"].fillna(0).tolist()

    # Add features not in the rewired list with score 0
    all_feature_ids = features["feature_id"].astype(str).tolist()
    present = set(all_ids)
    for fid in all_feature_ids:
        if fid not in present:
            all_ids.append(fid)
            scores.append(0.0)

    binary = [1 if fid in GROUND_TRUTH_REWIRED else 0 for fid in all_ids]

    if sum(binary) > 0 and sum(binary) < len(binary):
        auroc = roc_auc_score(binary, scores)
        ap    = average_precision_score(binary, scores)
        print(f"AUROC:              {auroc:.4f}  (random = 0.50)")
        print(f"Average Precision:  {ap:.4f}")
    else:
        print("AUROC not computable: all nodes are in one class.")

---

## 5. Explore the per-network triage

Each group-specific network also has a triage ranking based on `triage_score`, which combines network topology with phenotype association strength.

In [ ]:
# Find all priority_features.csv files produced
for p in sorted(outdir.rglob("priority_features.csv")):
    print(f"\n--- {p.relative_to(outdir)} ---")
    df = pd.read_csv(p)
    display(df.head(8))

**Reading triage_score:**

```
triage_score = topology_weight * topology_priority_score
             + selection_weight * selection_support_score
```

- `topology_priority_score` — composite of degree centrality, betweenness, eigenvector centrality, and community hub status. High = important network hub.
- `selection_support_score` — reflects how significantly the feature was associated with the phenotype during feature selection. High = strong phenotype signal.

Use `triage_score` to decide which features to follow up in the **group-specific** network context. Use `rewiring_score` to identify features with the most **changed** connectivity between groups.

---

## 6. Explore community structure

In [ ]:
for p in sorted(outdir.rglob("community_summary.csv")):
    if "triage" not in str(p):  # skip triage subdir copies
        continue
    print(f"\n--- {p.relative_to(outdir)} ---")
    df = pd.read_csv(p)
    display(df)

---

## 7. Try different parameters

Experiment with the parameters below to see how results change.

In [ ]:
# Example: stricter sign-switch filter
tmp2 = Path(tempfile.mkdtemp())
outdir2 = tmp2 / "output_strict"
outdir2.mkdir()

config2 = LeanConfig(
    **{
        **config.model_dump(),
        "OUTDIR": str(outdir2),
        "SIGN_SWITCH_MIN_R": 0.6,   # stricter — fewer false positives
        "TRIAGE_TOPOLOGY_WEIGHT": 0.5,
        "TRIAGE_SELECTION_WEIGHT": 0.5,
    }
)

run(config2)

p2 = next(outdir2.rglob("priority_rewired_nodes.csv"), None)
if p2:
    df2 = pd.read_csv(p2)
    print(f"Top rewired nodes with SIGN_SWITCH_MIN_R=0.6, TRIAGE_TOPOLOGY_WEIGHT=0.5:")
    display(df2.head(10))

---

## 8. Next steps

1. **Open `triage/rewiring_network.graphml` in [Cytoscape](https://cytoscape.org/).**
   - Colour edges by `state` (shared / ref_only / case_only)
   - Size nodes by `rewiring_score`
   - Filter to the top-15 rewired nodes to focus interpretation

2. **Focus on sign-switch nodes first** — these have `sign_switch_edges > 0` in `priority_rewired_nodes.csv`. A sign switch suggests the metabolite's relationship with its partners inverted direction, which often has strong biological meaning.

3. **Cross-reference with group-specific triage** — a node that is top-ranked in the Case-specific network *and* has high `rewiring_score` is a strong candidate for follow-up.

4. **Run the ground-truth benchmarks** to characterise detection power for your sample size:
   ```bash
   python -m tests.benchmark.rewiring_benchmark
   ```

5. **Adjust `SIGN_SWITCH_MIN_R`** if sign-switch recall is low. Lower values increase sensitivity but may detect spurious sign flips in small samples.

6. **Enable `LOUVAIN_STABILITY_CHECK: true`** to verify that Louvain community assignments are reproducible. A warning appears in the report if AMI < 0.8 across 10 random seeds.